# Sun Run 2026 — extract teams data

Reads the published Teams page (HTML with multiple `<pre>` blocks), parses it with `sunrun_parse.py`, builds `categories`, `teams`, and `runners`, then saves CSV files under `data/processed/`.

## Imports and paths

Libraries, source URL, output directory, and `sunrun_parse` (full parser for the Sportstats HTML format).

In [26]:
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

from sunrun_parse import extract_text_from_teams_html, parse_sunrun_teams_text

# Official results page (HTML with one <pre> block per division)
SOURCE_URL = "https://cdn-1.sportstats.one/SunRun2026_Teams.htm"

OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)


## Source format (why a dedicated parser)

The published file is **HTML** with **29 `<pre>` blocks** (one per division). Each block contains:

- **Category** — `10K Team Results - … Category`
- **Team** — `rank. H:MM:SS Team Name (average)` — the average in parentheses may be **MM:SS** or **H:MM:SS** (the old notebook required `H:MM:SS` for the average and dropped most team rows).
- **Runners** — **two athletes per line**: left column (finish rank within team, time, name) and right column (bib, time, name). Times may be **MM:SS** or **H:MM:SS**; the right column sometimes uses `(time)` and sometimes a bare time after the bib.

The previous line-only regex only matched times with **two colons**, so it effectively kept the **right-hand** runner only and mis-used **bib** as rank. Implementation: **`sunrun_parse.py`**.

In [27]:
# Parsing logic lives in sunrun_parse.py (concatenate all <pre> blocks, then state-machine parse).


## Load and normalize

Download HTML from `SOURCE_URL`, then concatenate every `<pre>…</pre>` block into one plain-text report (the site splits divisions across multiple `<pre>` tags).

In [28]:
resp = requests.get(SOURCE_URL, timeout=30)
resp.raise_for_status()
raw = resp.text
if not raw.strip():
    raise RuntimeError("Empty response from URL.")

text = extract_text_from_teams_html(raw)


## Parse

State machine (in `parse_sunrun_teams_text`): **category** → **team** header → **runner** lines until the next team or category. Each runner line yields one or two athletes (left column, then optional right column).

In [29]:
cats, teams, runners = parse_sunrun_teams_text(text)


## Build DataFrames

Merge runner rows with team metadata for filtering by category or team name.

In [30]:
categories_df = pd.DataFrame(cats).drop_duplicates("category_id")
teams_df = pd.DataFrame(teams)
runners_df = pd.DataFrame(runners)
if "runner_bib" in runners_df.columns:
    runners_df["runner_bib"] = runners_df["runner_bib"].astype("Int64")

if len(runners_df) and len(teams_df):
    runners_df = runners_df.merge(
        teams_df[["team_id", "category_id", "team_name", "category_rank"]],
        on="team_id",
    )


## Preview

Quick sanity check of row counts and first rows.

In [31]:
print(
    len(categories_df),
    "categories",
    len(teams_df),
    "teams",
    len(runners_df),
    "runners",
)
assert teams_df["team_id"].is_unique
assert categories_df["category_id"].is_unique
# Spot-check: first Accounting team should be Deloitte with a large roster
_row = teams_df.loc[(teams_df["category_id"] == 1) & (teams_df["category_rank"] == 1)]
if len(_row):
    _tid = int(_row["team_id"].iloc[0])
    print(
        "Accounting #1 team:",
        _row["team_name"].iloc[0],
        "runner rows:",
        int((runners_df["team_id"] == _tid).sum()),
    )

display(categories_df.head(), teams_df.head(), runners_df.head())


29 categories 683 teams 6699 runners


,category_id,category_name
0,1,Accounting Firms
1,2,Advertising/Marketing Services
2,3,Education
3,4,Engineering
4,5,Financial - Banking & Investment


,team_id,category_id,category_rank,team_total_time,team_total_seconds,team_name,team_avg_time,team_avg_seconds
0,1,1,15,8:03:46,29026,Team Clearline,1:00:29,3629
1,2,1,16,8:14:03,29643,Smythe in the Fast Lane,1:01:46,3706
2,3,1,17,9:20:36,33636,Rolfe Benson,1:10:05,4205
3,4,1,18,10:21:49,37309,AFFINITY ACCOUNTANTS LLP,1:17:44,4664
4,5,1,19,11:46:02,42362,Loewen Kruse CPA,1:28:16,5296


,team_id,runner_rank_in_team,runner_time,runner_seconds,runner_name,category_id,team_name,category_rank
0,1,10,1:11:53,4313,Angeline Martinez,1,Team Clearline,15
1,1,11,1:17:52,4672,Annelie Vistica,1,Team Clearline,15
2,1,12,1:17:53,4673,Dan Vistica,1,Team Clearline,15
3,1,4,1:00:46,3646,Jason Ghag,1,Team Clearline,15
4,1,13,1:25:54,5154,Jason Shin,1,Team Clearline,15


## Save

Write CSV files and print each file size.

In [32]:
def _human_bytes(n: int) -> str:
    if n < 1024:
        return f"{n} B"
    if n < 1024 * 1024:
        return f"{n / 1024:.1f} KB"
    return f"{n / 1024 / 1024:.1f} MB"


for stem, df in [("categories", categories_df), ("teams", teams_df), ("runners", runners_df)]:
    path = OUT / f"sunrun_{stem}.csv"
    df.to_csv(path, index=False)
    size = path.stat().st_size
    print(f"{path.name}: {_human_bytes(size)}")

print("Saved to", OUT)


sunrun_categories.csv: 724 B
sunrun_teams.csv: 36.0 KB
sunrun_runners.csv: 365.5 KB
Saved to data/processed
